# Olist 数据清洗与特征工程

**目标**：基于已加载的 9 张 Olist 表，完成清洗、特征提取、最终生成 `fact_order` 宽表。

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime

# ==== 连接已建好的 SQLite 数据库 ====
DB_PATH = "olist.db"
conn = sqlite3.connect(DB_PATH)

# 加载全部原始表
orders       = pd.read_sql("SELECT * FROM orders", conn)
order_items  = pd.read_sql("SELECT * FROM order_items", conn)
order_pay    = pd.read_sql("SELECT * FROM order_payments", conn)
order_rev    = pd.read_sql("SELECT * FROM order_reviews", conn)
products     = pd.read_sql("SELECT * FROM products", conn)
customers    = pd.read_sql("SELECT * FROM customers", conn)
sellers      = pd.read_sql("SELECT * FROM sellers", conn)
geo          = pd.read_sql("SELECT * FROM geolocation", conn)
cat_trans    = pd.read_sql("SELECT * FROM category_trans", conn)

print("All tables loaded.")
print(f"orders:       {len(orders):>10,}")
print(f"order_items:  {len(order_items):>10,}")
print(f"order_pay:    {len(order_pay):>10,}")
print(f"products:     {len(products):>10,}")
print(f"customers:    {len(customers):>10,}")
print(f"sellers:      {len(sellers):>10,}")

All tables loaded.
orders:           99,441
order_items:     112,650
order_pay:       103,886
products:         32,951
customers:        99,441
sellers:           3,095


---
## 1. 订单表 (orders) 处理

- 转为日期类型，提取年/月/季度/星期
- 计算物流天数
- 标记订单状态

In [2]:
# ---- 1.1 时间字段统一转为 datetime 类型 ----
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# ---- 1.2 提取时间维度 ----
purchase = orders['order_purchase_timestamp']  # 简写

orders['purchase_year']    = purchase.dt.year
orders['purchase_month']   = purchase.dt.month
orders['purchase_quarter'] = purchase.dt.quarter
orders['purchase_weekday'] = purchase.dt.day_name()          # Monday, Tuesday ...
orders['purchase_ym']      = purchase.dt.strftime('%Y-%m')   # 2023-07
orders['purchase_date']    = purchase.dt.date                # 纯日期，用于后续 merge

# ---- 1.3 计算物流天数 ----
orders['logistics_days'] = (
    orders['order_delivered_customer_date'] - 
    orders['order_purchase_timestamp']
).dt.days  # 转为整数天

# ---- 1.4 标记订单状态 ----
def map_order_status(s):
    if s == 'delivered':
        return '已完成'
    elif s == 'canceled':
        return '已取消'
    else:
        return '其他'

orders['order_status_cn'] = orders['order_status'].apply(map_order_status)

# 快速查看
print("=== 订单状态分布 ===")
print(orders['order_status_cn'].value_counts())
print(f"\n物流天数范围: {orders['logistics_days'].min():.0f} ~ {orders['logistics_days'].max():.0f} 天")
print(f"物流天数中位数: {orders['logistics_days'].median():.1f} 天")
orders[date_cols + ['purchase_year', 'purchase_month', 'purchase_quarter',
                     'purchase_weekday', 'logistics_days', 'order_status_cn']].head(3)

=== 订单状态分布 ===
已完成    96478
其他      2338
已取消      625
Name: order_status_cn, dtype: int64

物流天数范围: 0 ~ 209 天
物流天数中位数: 10.0 天


,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_year,purchase_month,purchase_quarter,purchase_weekday,logistics_days,order_status_cn
0,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,10,4,Monday,8.0,已完成
1,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018,7,3,Tuesday,13.0,已完成
2,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018,8,3,Wednesday,9.0,已完成


---
## 2. 订单商品明细表 (order_items) 处理

- 计算每个订单的总金额 (price + freight_value)
- 计算每个订单的商品数量

In [3]:
# ---- 2.1 按订单聚合 ----
order_agg = order_items.groupby('order_id').agg(
    order_item_count  = ('order_item_id', 'count'),      # 商品件数
    order_total_price = ('price',          'sum'),       # 商品总价
    order_freight     = ('freight_value',  'sum'),       # 总运费
).reset_index()

# 订单总金额 = 商品总价 + 运费
order_agg['order_total_amount'] = order_agg['order_total_price'] + order_agg['order_freight']

print(f"聚合后行数: {len(order_agg):,}  (原表: {len(order_items):,})")
order_agg.head(5)

聚合后行数: 98,666  (原表: 112,650)


,order_id,order_item_count,order_total_price,order_freight,order_total_amount
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,218.04


---
## 3. 客户表 (customers) 处理

- 统计每个客户的首次购买日期、最近购买日期、购买频次

In [4]:
# ---- 3.1 关联 orders 表获取购买时间 ----
cust_purchase = orders[['order_id', 'customer_id', 'order_purchase_timestamp']].merge(
    order_agg[['order_id', 'order_total_amount']],
    on='order_id',
    how='inner'
)

# ---- 3.2 按 customer_id 聚合 ----
customer_stats = cust_purchase.groupby('customer_id').agg(
    first_purchase_date  = ('order_purchase_timestamp', 'min'),   # 首次购买
    last_purchase_date   = ('order_purchase_timestamp', 'max'),   # 最近购买
    purchase_frequency   = ('order_id',                'count'),  # 购买频次
    total_spent          = ('order_total_amount',      'sum'),    # 累计消费
    avg_order_value      = ('order_total_amount',      'mean'),   # 单均消费
).reset_index()

print(f"客户数: {len(customer_stats):,}")
print(f"复购客户数: {(customer_stats['purchase_frequency'] > 1).sum():,}")
print(f"复购率: {(customer_stats['purchase_frequency'] > 1).mean():.2%}")
customer_stats.head(5)

客户数: 98,666
复购客户数: 0
复购率: 0.00%


,customer_id,first_purchase_date,last_purchase_date,purchase_frequency,total_spent,avg_order_value
0,00012a2ce6f8dcda20d059ce98491703,2017-11-14 16:08:26,2017-11-14 16:08:26,1,114.74,114.74
1,000161a058600d5901f007fab4c27140,2017-07-16 09:40:32,2017-07-16 09:40:32,1,67.41,67.41
2,0001fd6190edaaf884bcaf3d49edf079,2017-02-28 11:06:43,2017-02-28 11:06:43,1,195.42,195.42
3,0002414f95344307404f0ace7a26f1d5,2017-08-16 13:09:20,2017-08-16 13:09:20,1,179.35,179.35
4,000379cdec625522490c315e70c7a9fb,2018-04-02 13:42:17,2018-04-02 13:42:17,1,107.01,107.01


---
## 4. 产品表 (products) 处理

- 关联产品分类翻译表，统一使用英文分类名

In [5]:
# ---- 4.1 关联翻译表 ----
products = products.merge(
    cat_trans,
    on='product_category_name',
    how='left'  # 保留所有产品，翻译不到的分类填 NaN
)

# ---- 4.2 缺失分类补全 ----
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')

print(f"品类数（英文）: {products['product_category_name_english'].nunique()}")
print(f"缺失分类产品数: {(products['product_category_name_english'] == 'unknown').sum()}")
products[['product_id', 'product_category_name', 'product_category_name_english']].head(5)

品类数（英文）: 72
缺失分类产品数: 623


,product_id,product_category_name,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares


---
## 5. 生成宽表 fact_order

以 order_items 为骨架，LEFT JOIN 各维度表，输出一张完整的分析宽表。

In [6]:
# ==== 5.1 以 order_items 为事实表骨架 ====
fact = order_items.copy()

# ---- 5.2 关联 orders：下单日期、物流天数、订单状态 ----
order_dim = orders[[
    'order_id', 'customer_id',
    'order_purchase_timestamp', 'order_purchase_timestamp',
    'logistics_days', 'order_status_cn',
    'purchase_year', 'purchase_month', 'purchase_quarter', 'purchase_weekday', 'purchase_ym'
]].rename(columns={'order_purchase_timestamp': 'purchase_time'})

fact = fact.merge(order_dim, on='order_id', how='left')

# ---- 5.3 关联 order_agg：订单总金额、商品数量 ----
fact = fact.merge(order_agg, on='order_id', how='left')

# ---- 5.4 关联 payments：取首笔支付方式 ----
#     一笔订单可能有多条支付记录（分期），只取 payment_sequential == 1
first_payment = order_pay[order_pay['payment_sequential'] == 1][['order_id', 'payment_type', 'payment_installments', 'payment_value']]
fact = fact.merge(first_payment, on='order_id', how='left')

# ---- 5.5 关联 products：品类、重量、尺寸 ----
product_dim = products[['product_id', 'product_category_name_english',
                        'product_weight_g', 'product_length_cm',
                        'product_height_cm', 'product_width_cm']]
fact = fact.merge(product_dim, on='product_id', how='left')

# ---- 5.6 关联 sellers ----
seller_dim = sellers[['seller_id', 'seller_city', 'seller_state']]
fact = fact.merge(seller_dim, on='seller_id', how='left')

# ---- 5.7 关联 customers ----
customer_dim = customers[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']]
fact = fact.merge(customer_dim, on='customer_id', how='left')

# ---- 5.8 关联 customer_stats（RFM 指标） ----
fact = fact.merge(customer_stats, on='customer_id', how='left')

print(f"fact_order 行数: {len(fact):,}")
print(f"fact_order 列数: {len(fact.columns)}")
print(f"\n列名列表:\n{list(fact.columns)}")

fact_order 行数: 112,650
fact_order 列数: 39

列名列表:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'customer_id', 'purchase_time', 'purchase_time', 'logistics_days', 'order_status_cn', 'purchase_year', 'purchase_month', 'purchase_quarter', 'purchase_weekday', 'purchase_ym', 'order_item_count', 'order_total_price', 'order_freight', 'order_total_amount', 'payment_type', 'payment_installments', 'payment_value', 'product_category_name_english', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_city', 'seller_state', 'customer_unique_id', 'customer_city', 'customer_state', 'first_purchase_date', 'last_purchase_date', 'purchase_frequency', 'total_spent', 'avg_order_value']


---
## 6. 宽表质量检查与写入 SQLite

In [8]:
# ==== 6.1 缺失值检查 ====
missing = fact.isnull().sum()
missing_pct = (missing / len(fact) * 100).round(2)
missing_df = pd.DataFrame({'缺失数': missing, '占比%': missing_pct})
print("=== 各列缺失值 ===")
print(missing_df[missing_df['缺失数'] > 0].sort_values('缺失数', ascending=False))

# ==== 6.2 基本统计 ====
print(f"\n=== fact_order 摘要 ===")
print(f"行数: {len(fact):,}")
print(f"订单数: {fact['order_id'].nunique():,}")
print(f"客户数: {fact['customer_id'].nunique():,}")
print(f"产品数: {fact['product_id'].nunique():,}")
print(f"卖家数: {fact['seller_id'].nunique():,}")
print(f"时间范围: {fact['purchase_time'].min()} ~ {fact['purchase_time'].max()}")

# ==== 6.3 写入 SQLite ====
conn_write = sqlite3.connect(DB_PATH)
fact.to_sql('fact_order', conn_write, if_exists='replace', index=False)

# 建立常用索引
conn_write.execute("CREATE INDEX IF NOT EXISTS idx_fact_order_id ON fact_order(order_id)")
conn_write.execute("CREATE INDEX IF NOT EXISTS idx_fact_customer ON fact_order(customer_id)")
conn_write.execute("CREATE INDEX IF NOT EXISTS idx_fact_product ON fact_order(product_id)")
conn_write.execute("CREATE INDEX IF NOT EXISTS idx_fact_date   ON fact_order(purchase_ym)")
conn_write.execute("CREATE INDEX IF NOT EXISTS idx_fact_category ON fact_order(product_category_name_english)")
conn_write.commit()
conn_write.close()

print(f"\n[DONE] fact_order 已写入 {DB_PATH}")

=== 各列缺失值 ===
                       缺失数   占比%
logistics_days        2454  2.18
payment_type            92  0.08
payment_installments    92  0.08
payment_value           92  0.08
product_weight_g        18  0.02
product_length_cm       18  0.02
product_height_cm       18  0.02
product_width_cm        18  0.02

=== fact_order 摘要 ===
行数: 112,650
订单数: 98,666
客户数: 98,666
产品数: 32,951
卖家数: 3,095
时间范围: purchase_time   2016-09-04 21:15:19
purchase_time   2016-09-04 21:15:19
dtype: datetime64[ns] ~ purchase_time   2018-09-03 09:06:57
purchase_time   2018-09-03 09:06:57
dtype: datetime64[ns]


OperationalError: duplicate column name: purchase_time

---
## 7. 最终宽表字段一览

| 分类 | 字段 | 来源 |
|------|------|------|
| **订单维度** | order_id, order_status_cn, logistics_days, purchase_time | orders |
| **时间维度** | purchase_year, purchase_month, purchase_quarter, purchase_weekday, purchase_ym | orders |
| **商品维度** | product_id, product_category_name_english, product_weight_g, product_length/height/width_cm | products |
| **金额维度** | price, freight_value, order_item_count, order_total_amount | order_items |
| **支付维度** | payment_type, payment_installments, payment_value | order_payments |
| **客户维度** | customer_id, customer_unique_id, customer_city/state | customers |
| **RFM 指标** | first_purchase_date, last_purchase_date, purchase_frequency, total_spent, avg_order_value | 计算字段 |
| **卖家维度** | seller_id, seller_city, seller_state | sellers |